In [1]:
from numpy import mean
from numpy import std
from matplotlib import pyplot
from sklearn.model_selection import KFold
from keras.datasets import mnist
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Conv2D
from keras.layers import MaxPooling2D
from keras.layers import Dense
from keras.layers import Flatten
from keras.optimizers import SGD
from skimage.feature import hog
import numpy as np
import cv2
from keras.preprocessing.image import load_img
from keras.preprocessing.image import img_to_array
from keras.models import load_model

Using TensorFlow backend.


In [2]:
def load_dataset():
	# load dataset
	(trainX, trainY), (testX, testY) = mnist.load_data()
	# reshape dataset to have a single channel
	trainX = trainX.reshape((trainX.shape[0], 28, 28, 1))
	testX = testX.reshape((testX.shape[0], 28, 28, 1))
	# one hot encode target values
	trainY = to_categorical(trainY)
	testY = to_categorical(testY)
	return trainX, trainY, testX, testY

# scale pixels
def prep_pixels(train, test):
	# convert from integers to floats
	train_norm = train.astype('float32')
	test_norm = test.astype('float32')
	# normalize to range 0-1
	train_norm = train_norm / 255.0
	test_norm = test_norm / 255.0
	# return normalized images
	return train_norm, test_norm

# define cnn model
def define_model():
	model = Sequential()
	model.add(Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_uniform', input_shape=(28, 28, 1)))
	model.add(MaxPooling2D((2, 2)))
	model.add(Flatten())
	model.add(Dense(100, activation='relu', kernel_initializer='he_uniform'))
	model.add(Dense(10, activation='softmax'))
	# compile model
	opt = SGD(lr=0.01, momentum=0.9)
	model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
	return model

# evaluate a model using k-fold cross-validation
def evaluate_model(model, dataX, dataY, n_folds=5):
	scores, histories = list(), list()
	# prepare cross validation
	kfold = KFold(n_folds, shuffle=True, random_state=1)
	# enumerate splits
	for train_ix, test_ix in kfold.split(dataX):
		# select rows for train and test
		trainX, trainY, testX, testY = dataX[train_ix], dataY[train_ix], dataX[test_ix], dataY[test_ix]
		# fit model
		history = model.fit(trainX, trainY, epochs=10, batch_size=32, validation_data=(testX, testY), verbose=0)
		# evaluate model
		_, acc = model.evaluate(testX, testY, verbose=0)
		print('> %.3f' % (acc * 100.0))
		# stores scores
		scores.append(acc)
		histories.append(history)
	return scores, histories, model

# plot diagnostic learning curves
def summarize_diagnostics(histories):
	for i in range(len(histories)):
		# plot loss
		pyplot.subplot(211)
		pyplot.title('Cross Entropy Loss')
		pyplot.plot(histories[i].history['loss'], color='blue', label='train')
		pyplot.plot(histories[i].history['val_loss'], color='orange', label='test')
		# plot accuracy
		pyplot.subplot(212)
		pyplot.title('Classification Accuracy')
		pyplot.plot(histories[i].history['accuracy'], color='blue', label='train')
		pyplot.plot(histories[i].history['val_accuracy'], color='orange', label='test')
	pyplot.show()

# summarize model performance
def summarize_performance(scores):
	# print summary
	print('Accuracy: mean=%.3f std=%.3f, n=%d' % (mean(scores)*100, std(scores)*100, len(scores)))
	# box and whisker plots of results
	pyplot.boxplot(scores)
	pyplot.show()

In [3]:
"""trainX, trainY, testX, testY = load_dataset()
# prepare pixel data
trainX, testX = prep_pixels(trainX, testX)
# define model
model = define_model()
# fit model
model.fit(trainX, trainY, epochs=10, batch_size=8, verbose=0)
# save model
model.save('final_model.h5')"""

"trainX, trainY, testX, testY = load_dataset()\n# prepare pixel data\ntrainX, testX = prep_pixels(trainX, testX)\n# define model\nmodel = define_model()\n# fit model\nmodel.fit(trainX, trainY, epochs=10, batch_size=8, verbose=0)\n# save model\nmodel.save('final_model.h5')"

In [4]:
def img_test(filename,model_name):
    model = load_model(model_name)
    # Loading the image
    im = cv2.imread(file_name)
    # pyplot.imshow(im)
    # pyplot.show()

    # im = cv2.rotate(im,cv2.ROTATE_90_COUNTERCLOCKWISE)

    # RGB to Gray conversion
    im_gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    # pyplot.imshow(im_gray)
    # pyplot.show()

    # Rotating image
    # im_gray = cv2.rotate(im_gray,cv2.ROTATE_90_COUNTERCLOCKWISE)

    # Applying Gaussian Blur
    im_gray = cv2.GaussianBlur(im_gray, (5, 5), 0)
    # pyplot.imshow(im_gray)
    # pyplot.show()

    # Thresholding the image
    ret, im_th = cv2.threshold(im_gray, 127, 255, cv2.THRESH_BINARY_INV)
    # pyplot.imshow(im_th)
    # pyplot.show()
    # Finding contour in image
    ctrs, hier = cv2.findContours(im_th.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Getting Bounding Boxes
    rects = [cv2.boundingRect(ctr) for ctr in ctrs]

    i= 0
    digits = []
    for rect in rects:
        rectangle = cv2.rectangle(im, (rect[0],rect[1]), (rect[0]+rect[2],rect[1]+rect[3]) , (0,255,0), 3 )

        leng = int(rect[3]+20)
        # print("leng:",leng)
        pt1 = int(rect[1] + rect[3] // 2 - leng // 2)
        pt2 = int(rect[0] + rect[2] // 2 - leng // 2)
        roi = im_th[pt1:pt1+leng, pt2:pt2+leng]

        roi_res = cv2.resize(roi, (28,28) , interpolation=cv2.INTER_AREA)
        roi_dilated = cv2.dilate(roi_res, (3,3))

        # roi_hog_fd = hog(roi_dilated, orientations=9, pixels_per_cell=(14,14), cells_per_block=(1,1), visualize=False)
        # roi_hog_fd.shape

        img = roi_dilated
        img = img_to_array(img)
        img = img.reshape(1,28,28,1)
        img = img.astype('float32')
        img = img / 255.0
        #pyplot.subplot(330 + 1 + i)
        #pyplot.imshow(roi_dilated, cmap=pyplot.get_cmap('gray'))
        #i = i + 1
        digit = model.predict_classes(img)
        cv2.putText(im, str(int(digit)), (rect[0], rect[1]),cv2.FONT_HERSHEY_DUPLEX, 2, (255, 255, 255), 3)
        digits.append(digit[0])
    # pyplot.show()
    new_name = "Images_Output/img"+''.join([str(elem) for elem in digits])+".jpg"
    pyplot.imsave(new_name,im)
    return im,digits

In [8]:
mdl = load_model("final_model.h5")
mdl.summary()

Model: "sequential_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d_1 (Conv2D)            (None, 26, 26, 32)        320       
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 13, 13, 32)        0         
_________________________________________________________________
flatten_1 (Flatten)          (None, 5408)              0         
_________________________________________________________________
dense_1 (Dense)              (None, 100)               540900    
_________________________________________________________________
dense_2 (Dense)              (None, 10)                1010      
Total params: 542,230
Trainable params: 542,230
Non-trainable params: 0
_________________________________________________________________


In [6]:
from os import walk
mypath = "/home/nishat/Desktop/ML_Project_3/Images/"
files = []
for (dirpath, dirnames, filenames) in walk(mypath):
    files.extend(filenames)
    break

model_name = 'final_model.h5'
for file in files:
    file_name = 'Images/'+str(file)
    # print(file_name)
    img, digits = img_test(file_name,model_name)
    # print("digits:",digits)
    print(file_name," is Done!")

Images/img5.jpg  is Done!
Images/img2.jpg  is Done!
Images/img4.jpg  is Done!
Images/img6.jpg  is Done!
Images/img1.jpg  is Done!
Images/img3.jpg  is Done!
